[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thanathi/Astro_Gateway_2026/blob/main/Notebooks/03_TimeSeries.ipynb)

# 03 · อนุกรมเวลา: ดาวแปรแสงและดาวเคราะห์นอกระบบ
**Astro Gateway** · คาบ 13.00–14.15 น.

**กราฟแสง (light curve)** คือความสว่างของวัตถุที่เปลี่ยนไปตามเวลา จากกราฟแสงเราหาได้ว่า
- ดาวแปรแสง **เต้นเป็นจังหวะคาบเท่าไร** → ใช้หาระยะทาง (ดาวเซเฟอิด)
- ดาวเคราะห์ **โคจรรอบละกี่วัน และใหญ่แค่ไหน** → จากการที่มันผ่านหน้าดาวแม่

> 📡 **ข้อมูลที่ใช้:** ส่วน A เป็นกราฟแสง **จริง** ของดาวเซเฟอิด OGLE-LMC-CEP-0728 ในกาแล็กซีเมฆแมเจลแลนใหญ่
> จากโครงการ OGLE (กล้อง 1.3 เมตร หอดูดาว Las Campanas ประเทศชิลี) ส่วน B เป็นกราฟแสง **จริง** ของดาว HD 209458
> ที่มีดาวเคราะห์ผ่านหน้า จากกล้องโทรทรรศน์อวกาศ TESS (ทุก 2 นาที, sector 56 ปี ค.ศ. 2022)

> ✏️ **ช่อง `____` ในโค้ดคือส่วนที่ต้องเติมเอง** (สูตรอยู่ในข้อความด้านบนหรือในสไลด์) — ถ้ารันแล้วเจอ `NameError: name '____' is not defined` แปลว่ายังเติมไม่ครบ
>
> 📁 ข้อมูลอยู่ที่ [github.com/thanathi/Astro_Gateway_2026](https://github.com/thanathi/Astro_Gateway_2026) โฟลเดอร์ Data —
> โน้ตบุ๊กจะดาวน์โหลดไฟล์ที่ต้องใช้ให้อัตโนมัติ ไม่ต้องตั้งค่าอะไร (ใช้ไฟล์ของตัวเองได้ด้วยการเปลี่ยนชื่อไฟล์ในโค้ด)
> 💾 เก็บงานของตัวเอง: **File → Save a copy in Drive** ก่อนเริ่มทำ

In [ ]:
#@title 🔧 ฟังก์ชันช่วย — กดรันเซลล์นี้ก่อน (ไม่ต้องแก้ไข) { display-mode: "form" }
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
COURSE = "Astro_Gateway_2026"     # ชื่อโฟลเดอร์รายวิชา (GitHub: thanathi/Astro_Gateway_2026)
DATA_URL = "https://raw.githubusercontent.com/thanathi/Astro_Gateway_2026/main/Data/"   # ข้อมูลทั้งหมดบน GitHub · None = ใช้โฟลเดอร์ใน Google Drive แทน
DATA_DIRS = [".", "Data", "../Data", "/content/Data"]


def find_course(root="/content/drive"):
    """หาโฟลเดอร์ COURSE ใน Google Drive: ชั้นบนสุดของ MyDrive, ลึกลงไป 1-2 ชั้น หรือใน Shared drives
    เลือกโฟลเดอร์ที่มี Data อยู่ข้างใน (กันกรณีแตกไฟล์ zip แล้วได้โฟลเดอร์ซ้อนกัน 2 ชั้น)"""
    import glob, os
    hits = []
    for pat in ("MyDrive/{c}", "MyDrive/*/{c}", "MyDrive/*/*/{c}", "Shareddrives/*/{c}"):
        hits += sorted(glob.glob(f"{root}/{pat.format(c=COURSE)}"))
    good = [h for h in hits if os.path.isdir(os.path.join(h, "Data"))]
    return (good or hits or [None])[0]


def connect_drive():
    """ใน Colab: ถ้าใช้ข้อมูลจาก GitHub (DATA_URL) ไม่ต้องทำอะไร — ไฟล์จะดาวน์โหลดเองเมื่อใช้
    ถ้า DATA_URL = None: เชื่อม Google Drive แล้วย้ายไปที่โฟลเดอร์ Notebooks ของรายวิชา ให้ ../Data ใช้ได้
    นอก Colab ไม่ทำอะไร"""
    import os
    try:
        from google.colab import drive
    except ImportError:
        return
    if DATA_URL:
        print("ข้อมูลจะดาวน์โหลดจาก GitHub ให้อัตโนมัติเมื่อใช้ (ไม่ต้องเชื่อม Google Drive)")
        return
    if not os.path.isdir("/content/drive/MyDrive"):
        drive.mount("/content/drive")
    course = find_course()
    if course is None:
        print(f"⚠️ ไม่พบโฟลเดอร์ {COURSE} ใน Google Drive — โน้ตบุ๊กจะดาวน์โหลดข้อมูลจากคลังข้อมูลต้นทางแทน")
        print(f"   (คัดลอกโฟลเดอร์ {COURSE} ไว้ใน My Drive ตามวิธีใน README.txt แล้วรันเซลล์นี้ใหม่)")
        return
    nb = os.path.join(course, "Notebooks")
    os.chdir(nb if os.path.isdir(nb) else course)
    DATA_DIRS.insert(0, os.path.join(course, "Data"))
    print(f"เชื่อม Google Drive แล้ว: ใช้ข้อมูลจาก {os.path.join(course, 'Data')}")


def data_path(rel, url=None):
    """หาไฟล์ข้อมูล rel (เช่น "SDSS/spec-0531-52028-0573.fits"): ดูในโฟลเดอร์ Data ก่อน
    (หรือไฟล์ที่อัปโหลดในแถบ Files ของ Colab) ถ้าไม่มี ให้ดาวน์โหลดจาก url (คลังข้อมูลต้นทาง)"""
    import os
    name = os.path.basename(rel)
    for base in DATA_DIRS:
        for p in (os.path.join(base, rel), os.path.join(base, name)):
            if os.path.exists(p):
                return p
    for p in (os.path.join("/content", name),):                 # ไฟล์ที่ดาวน์โหลด/อัปโหลดไว้แล้วใน Colab
        if os.path.exists(p):
            return p
    # เขียนไฟล์ที่ดาวน์โหลดไว้ที่ /content ใน Colab (โฟลเดอร์ที่แชร์แบบดูอย่างเดียวเขียนไม่ได้)
    out = os.path.join("/content" if os.path.isdir("/content") else ".", name)
    if DATA_URL and not os.path.isdir("/content"):          # นอก Colab: ไม่ดาวน์โหลดทับโฟลเดอร์ทำงาน
        out = os.path.join(os.path.expanduser("~"), ".astro_gateway_cache", name)
        os.makedirs(os.path.dirname(out), exist_ok=True)
    for src in ([DATA_URL + rel] if DATA_URL else []) + ([url] if url else []):
        try:
            import requests
            r = requests.get(src, timeout=(10, 60))
            r.raise_for_status()
            with open(out, "wb") as fh:
                fh.write(r.content)
            print(f"ดาวน์โหลด {name} จาก {src.split('/')[2]} แล้ว ({len(r.content) / 1e6:.1f} MB)")
            return out
        except Exception as e:
            print(f"  ดาวน์โหลดจาก {src.split('/')[2]} ไม่ได้ ({type(e).__name__})")
    raise FileNotFoundError(f"ไม่พบไฟล์ {name} — ตรวจการเชื่อมต่ออินเทอร์เน็ต แล้วรันเซลล์นี้ใหม่ "
                            f"(หรือดาวน์โหลดไฟล์จาก github.com/thanathi/{COURSE} โฟลเดอร์ Data/{os.path.dirname(rel) or '.'} "
                            "แล้วอัปโหลดที่แถบ Files)")






def synthetic_transit(period=3.5247, rp_rs=0.12, t0=1.2, duration_h=3.0, days=27.4,
                      cadence_min=10, noise=0.0015, seed=4):
    """กราฟแสงดาวที่มีดาวเคราะห์ผ่านหน้า แบบกล้อง TESS 1 sector -> (เวลา [วัน], ฟลักซ์ปรับเป็น 1, error)"""
    rng = np.random.default_rng(seed)
    t = np.arange(0, days, cadence_min / 1440)
    t = t[(t < 13.2) | (t > 14.4)]                          # ช่องว่างกลาง sector (ส่งข้อมูลกลับโลก)
    depth, half, ing = rp_rs ** 2, duration_h / 48, 0.4 / 24
    x = np.abs((t - t0 + period / 2) % period - period / 2)  # ระยะเวลาห่างจากกึ่งกลางการผ่านหน้า
    flux = 1 - depth * np.clip((half - x) / ing, 0, 1)      # รูปสี่เหลี่ยมคางหมู
    err = np.full_like(t, noise)
    return t, flux + rng.normal(0, err), err


def load_lightcurve(path):
    """อ่านกราฟแสงจาก CSV (time, flux หรือ mag[, error]), ไฟล์ตัวเลขไม่มีหัวตารางแบบ OGLE (เวลา แมกนิจูด error)
    หรือ FITS ของ TESS/Kepler -> คืนค่า (เวลา, ค่าความสว่าง, error, ชนิด 'flux' หรือ 'mag')"""
    path = str(path)
    with open(path, "rb") as fh:
        first = fh.readline().decode("ascii", "ignore").replace(",", " ").split()
    try:                                                  # บรรทัดแรกเป็นตัวเลขล้วน = ไม่มีหัวตาราง (OGLE)
        [float(v) for v in first]
        numeric = len(first) >= 2
    except ValueError:
        numeric = False
    if numeric:
        d = np.loadtxt(path, delimiter="," if path.lower().endswith(".csv") else None, ndmin=2)
        e = d[:, 2] if d.shape[1] > 2 else np.full(len(d), np.nan)
        return d[:, 0], d[:, 1], e, "mag"
    if path.lower().endswith((".fits", ".fit", ".fits.gz")):
        from astropy.io import fits
        with fits.open(path) as hdul:
            d = hdul[1].data
            names = [n.upper() for n in d.columns.names]
            fcol = next(c for c in ("PDCSAP_FLUX", "SAP_FLUX", "FLUX") if c in names)
            t = np.asarray(d["TIME"], float)
            f = np.asarray(d[fcol], float)
            e = np.asarray(d[fcol + "_ERR"], float) if fcol + "_ERR" in names else np.full_like(f, np.nan)
            good = np.isfinite(t) & np.isfinite(f)
            if fcol == "FLUX":                            # K2 แบบ EVEREST: ตัดช่วงแรก + ธงคุณภาพที่ใช้ไม่ได้
                bad = sum(2 ** (k - 1) for k in (1, 2, 3, 4, 5, 6, 7, 8, 9, 11, 12, 13, 14, 16, 17))
                good &= (np.arange(len(t)) > 100) & ((np.asarray(d["QUALITY"]) & bad) == 0)
            elif "QUALITY" in names:
                good &= np.asarray(d["QUALITY"]) == 0
            med = np.nanmedian(f[good])
            return t[good], f[good] / med, e[good] / med, "flux"
    df = pd.read_csv(path, sep=None, engine="python")
    low = {c.lower().strip(): c for c in df.columns}
    tcol = next(low[k] for k in ("time", "t", "bjd", "hjd", "jd", "mjd", "btjd", "date") if k in low)
    kind = "mag" if any(k in low for k in ("mag", "magnitude", "vmag")) else "flux"
    ycol = next(low[k] for k in (("mag", "magnitude", "vmag") if kind == "mag" else ("flux", "rel_flux", "f")) if k in low)
    ecol = next((low[k] for k in ("error", "err", "mag_err", "flux_err", "sigma", "uncertainty") if k in low), None)
    df = df.dropna(subset=[tcol, ycol])
    e = df[ecol].to_numpy(float) if ecol else np.full(len(df), np.nan)
    return df[tcol].to_numpy(float), df[ycol].to_numpy(float), e, kind


def clean_lightcurve(t, y, e=None, nsig=5, window=25):
    """ตัดจุดที่กระโดดผิดปกติ (เช่น ช่วงที่กล้องสั่นหรือมีแสงรบกวน) ออกจากกราฟแสง
    เทียบแต่ละจุดกับค่ามัธยฐานของจุดข้างเคียง window จุด แล้วตัดจุดที่ห่างเกิน nsig เท่าของความกระจาย"""
    order = np.argsort(t)
    t, y = np.asarray(t, float)[order], np.asarray(y, float)[order]
    e = np.full_like(y, np.nan) if e is None else np.asarray(e, float)[order]
    ok = np.isfinite(t) & np.isfinite(y)
    t, y, e = t[ok], y[ok], e[ok]
    trend = pd.Series(y).rolling(window, center=True, min_periods=1).median().to_numpy()
    r = y - trend
    mad = 1.4826 * np.median(np.abs(r - np.median(r)))
    if not mad > 0:                       # กราฟเรียบมากจนค่ามัธยฐานข้างเคียงเท่ากับจุดนั้นเอง (r = 0) → ใช้ความต่างของจุดติดกันแทน
        mad = 1.4826 * np.median(np.abs(np.diff(y))) / np.sqrt(2)
    if not mad > 0:
        return t, y, e
    keep = np.abs(r) <= nsig * mad
    return t[keep], y[keep], e[keep]


def detrend(t, f, window=101, nsig=3):
    """แยกการเปลี่ยนแปลงช้า ๆ (จากตัวดาวหรือกล้อง) ออกจากการหรี่สั้น ๆ ของการผ่านหน้า
    ฟิตเส้นโค้งเรียบ (Savitzky-Golay, window จุด) ซ้ำ ๆ โดยไม่นับจุดที่หลุด แล้วตัดเฉพาะจุดที่สว่างผิดปกติ
    -> (เวลา, ฟลักซ์หลังหารด้วยแนวโน้ม, แนวโน้ม)"""
    from scipy.signal import savgol_filter
    t, f = np.asarray(t, float), np.asarray(f, float)
    keep = np.ones(len(t), bool)
    for _ in range(10):
        trend = savgol_filter(np.interp(t, t[keep], f[keep]), window, 3)
        r = f - trend
        sig = np.sqrt(np.mean(r[keep] ** 2))
        new = np.abs(r) < nsig * sig
        if new.sum() == keep.sum():
            break
        keep = new
    good = r < nsig * sig                                 # ตัดแค่จุดที่สว่างเกิน (การผ่านหน้าทำให้มืดลง)
    return t[good], f[good] / trend[good], trend[good]

connect_drive()
print("พร้อมใช้งาน ✓")

In [ ]:
from astropy.timeseries import LombScargle, BoxLeastSquares

## ตั้งค่าไฟล์ข้อมูล

In [ ]:
# ไฟล์กราฟแสง: CSV (time, flux หรือ mag), ไฟล์ OGLE (.dat) หรือ FITS ของ TESS/Kepler
OGLE_URL = "https://www.astrouw.edu.pl/ogle/ogle4/OCVS/lmc/cep/phot/I/"     # คลังข้อมูลสาธารณะของ OGLE
MY_NUMBER = 0          # 🔧 ส่วน A: ใส่หมายเลขของคุณ (1–54) · 0 = ดาวตัวอย่าง OGLE-LMC-CEP-0728
CLASS_STARS = [        # ดาวเซเฟอิดของทั้งห้อง คนละ 1 ดวง (OGLE-IV, เมฆแมเจลแลนใหญ่)
    "OGLE-LMC-CEP-1061", "OGLE-LMC-CEP-2365", "OGLE-LMC-CEP-4291", "OGLE-LMC-CEP-2772",
    "OGLE-LMC-CEP-1165", "OGLE-LMC-CEP-2878", "OGLE-LMC-CEP-2309", "OGLE-LMC-CEP-1253",
    "OGLE-LMC-CEP-4518", "OGLE-LMC-CEP-1528", "OGLE-LMC-CEP-2368", "OGLE-LMC-CEP-4475",
    "OGLE-LMC-CEP-0199", "OGLE-LMC-CEP-3554", "OGLE-LMC-CEP-2600", "OGLE-LMC-CEP-2242",
    "OGLE-LMC-CEP-3793", "OGLE-LMC-CEP-2852", "OGLE-LMC-CEP-4281", "OGLE-LMC-CEP-3018",
    "OGLE-LMC-CEP-1870", "OGLE-LMC-CEP-2635", "OGLE-LMC-CEP-2449", "OGLE-LMC-CEP-3923",
    "OGLE-LMC-CEP-1866", "OGLE-LMC-CEP-2127", "OGLE-LMC-CEP-3317", "OGLE-LMC-CEP-2028",
    "OGLE-LMC-CEP-0757", "OGLE-LMC-CEP-1214", "OGLE-LMC-CEP-3281", "OGLE-LMC-CEP-0366",
    "OGLE-LMC-CEP-1252", "OGLE-LMC-CEP-0114", "OGLE-LMC-CEP-4625", "OGLE-LMC-CEP-2133",
    "OGLE-LMC-CEP-3163", "OGLE-LMC-CEP-2102", "OGLE-LMC-CEP-2922", "OGLE-LMC-CEP-1578",
    "OGLE-LMC-CEP-1166", "OGLE-LMC-CEP-0727", "OGLE-LMC-CEP-1180", "OGLE-LMC-CEP-0545",
    "OGLE-LMC-CEP-2023", "OGLE-LMC-CEP-1086", "OGLE-LMC-CEP-2979", "OGLE-LMC-CEP-1080",
    "OGLE-LMC-CEP-2949", "OGLE-LMC-CEP-2944", "OGLE-LMC-CEP-0935", "OGLE-LMC-CEP-0360",
    "OGLE-LMC-CEP-2176", "OGLE-LMC-CEP-4634",
]
STAR = CLASS_STARS[MY_NUMBER - 1] if MY_NUMBER > 0 else "OGLE-LMC-CEP-0728"
VARIABLE_FILE = data_path(f"Cepheid/class/{STAR}_I.dat" if MY_NUMBER > 0 else f"Cepheid/{STAR}.dat", url=OGLE_URL + STAR + ".dat")
MAST_URL = "https://mast.stsci.edu/api/v0.1/Download/file?uri=mast:TESS/product/"   # คลังข้อมูลของกล้องอวกาศ (MAST)
LC = "tess2022244194134-s0056-0000000420814525-0243-s_lc.fits"   # HD 209458 (TESS sector 56)
TRANSIT_FILE = data_path("TESS/" + LC, url=MAST_URL + LC)
print(f"ดาวเซเฟอิดของคุณ: {STAR} →", VARIABLE_FILE)
print("ไฟล์ดาวเคราะห์:", TRANSIT_FILE)

# ส่วน A: ดาวแปรแสงเซเฟอิด

In [ ]:
t, y, yerr, kind = load_lightcurve(VARIABLE_FILE)

if kind == "flux":                       # แปลงฟลักซ์เป็นแมกนิจูดสัมพัทธ์
    yerr = 1.0857 * yerr / y
    y = -2.5 * np.log10(y / np.nanmedian(y))

t = t - t.min()                          # เริ่มนับเวลาจาก 0
plt.figure(figsize=(12, 3.5))
plt.errorbar(t, y, yerr, fmt="o", ms=3, color="k", ecolor="lightgray")
plt.gca().invert_yaxis()
plt.xlabel("time (days)"); plt.ylabel("I magnitude"); plt.title(f"{STAR}: raw light curve"); plt.show()
print(f"{STAR}: {len(t)} จุด ตลอด {t.max():.0f} วัน ({t.max() / 365.25:.1f} ปี)")

ทุกคนได้ดาวเซเฟอิดคนละดวงในกาแล็กซี **เมฆแมเจลแลนใหญ่** (Large Magellanic Cloud, LMC)
กาแล็กซีบริวารของทางช้างเผือกที่มองเห็นได้ด้วยตาเปล่าจากซีกฟ้าใต้ — โครงการ OGLE ถ่ายภาพในแถบสี I (อินฟราเรดใกล้) หลายร้อยครั้งตลอดหลายปี
(ยังไม่ได้หมายเลข? ใช้ `MY_NUMBER = 0` = ดาวตัวอย่าง OGLE-LMC-CEP-0728 ไปก่อน แล้วค่อยเปลี่ยนในเซลล์ "ตั้งค่าไฟล์ข้อมูล" และรันใหม่)

🔎 ดูด้วยตาออกไหมว่ากี่วันต่อรอบ? ยาก! เพราะเราสังเกตได้แค่คืนละครั้ง (และบางคืนฟ้าปิด)
ลองซูมดูช่วงสั้น ๆ: เพิ่ม `plt.xlim(250, 300)` ก่อน `plt.show()`

## A1. หาคาบด้วย Lomb–Scargle periodogram
ลองทุกคาบที่เป็นไปได้ แล้ววัดว่าคาบไหนอธิบายข้อมูลได้ดีที่สุด — ยอดสูงสุด = คาบที่น่าจะเป็นจริง

In [ ]:
ls = LombScargle(t, y, yerr)
freq, power = ls.autopower(minimum_frequency=1/30, maximum_frequency=2, samples_per_peak=20)
best = np.argmax(power)                  # ตำแหน่งของยอดสูงสุด
period = ____                  # ✏️ คาบ = 1 / ความถี่

plt.figure(figsize=(10, 3.5))
plt.plot(1 / freq, power, lw=1)
plt.axvline(period, color="tab:orange", ls="--")
plt.xscale("log")
plt.xlabel("period (days)"); plt.ylabel("power"); plt.title("Lomb-Scargle periodogram"); plt.show()
print(f"คาบที่ดีที่สุด = {period:.3f} วัน")

> 💡 ยอดสูงอื่น ๆ เป็น **ภาพลวง (alias)** ที่เกิดจากการสังเกตแค่ตอนกลางคืนทุก ๆ 1 วัน (ความถี่ 1/P ± 1 รอบต่อวัน)
> ลองพับกราฟด้วยคาบเหล่านั้นในขั้น A2 ดู แล้วจะรู้ว่าคาบไหนคือคาบจริง

## A2. พับกราฟแสง (phase folding)
นำทุกจุดมาวางซ้อนกันตามตำแหน่งในรอบ: **phase = (เวลา / คาบ) − ส่วนจำนวนเต็ม**

In [ ]:
def fold_plot(period):
    phase = ____          # ✏️ ส่วนเศษของ (เวลา / คาบ) — ใช้ np.mod(x, 1)
    plt.figure(figsize=(8, 4))
    plt.errorbar(np.r_[phase, phase + 1], np.r_[y, y], np.r_[yerr, yerr], fmt="o", ms=3, color="k", ecolor="lightgray")
    plt.gca().invert_yaxis()
    plt.xlabel("phase"); plt.ylabel("magnitude"); plt.title(f"folded at P = {period:.3f} d"); plt.show()

fold_plot(period)

🔧 **ลองเอง:** พับด้วยคาบผิด เช่น `fold_plot(1.23)` หรือ `fold_plot(period * 1.05)` แล้วดูว่ากราฟเป็นอย่างไร

In [ ]:
# เขียนโค้ดของคุณตรงนี้

## A3. ระยะทางจากความสัมพันธ์คาบ-กำลังส่องสว่าง (กฎของลีวิตต์)
ดาวเซเฟอิดที่ **คาบยาวกว่าจะสว่างจริงมากกว่า** — ค้นพบโดย Henrietta Leavitt (ค.ศ. 1912)

ความสัมพันธ์ในแถบสี I (ปรับเทียบจากดาวเซเฟอิดในทางช้างเผือกที่วัดพารัลแลกซ์ด้วยกล้องฮับเบิล; Benedict et al. 2007):

**M_I = −2.81 (log₁₀ P − 1) − 4.76**  (P เป็นวัน)

แล้วใช้ **ระยะทาง = 10^((m − M + 5) / 5)** พาร์เซก

In [ ]:
mean_mag = np.mean(y)            # แมกนิจูดเฉลี่ยในแถบ I (ถ้าเป็นแมกนิจูดสัมพัทธ์จากฟลักซ์ ให้ใส่ค่าจริงแทน)
A_I = 0.0                        # 🔧 การหรี่ของแสงจากฝุ่นในแถบ I (ทิศทาง LMC ≈ 0.15)

M_I = ____          # ✏️ กฎของลีวิตต์
d_pc = ____       # ✏️ ระยะทาง (พาร์เซก) — หัก A_I ออกจาก m ก่อน
amp = np.percentile(y, 98) - np.percentile(y, 2)
print(f"คาบ {period:.3f} วัน | แอมพลิจูด ≈ {amp:.2f} แมกนิจูด")
print(f"แมกนิจูดสัมบูรณ์ M_I = {M_I:.2f} | แมกนิจูดเฉลี่ย m = {mean_mag:.2f}")
print(f"ระยะทาง ≈ {d_pc / 1000:,.1f} กิโลพาร์เซก ≈ {d_pc * 3.26:,.0f} ปีแสง")

🔎 ระยะทางไปเมฆแมเจลแลนใหญ่ที่วัดด้วยวิธีอื่น (ดาวคู่อุปราคา) คือ **49.6 กิโลพาร์เซก** (≈ 162,000 ปีแสง)
— ลองใส่ `A_I = 0.15` แล้วผลใกล้ขึ้นไหม? ทำไมฝุ่นทำให้เราคิดว่าดาวอยู่ **ไกลเกินจริง**?

ระยะทางจากดาวดวงเดียวคลาดได้หลายกิโลพาร์เซก — ถ้ารวมดาวของทั้งห้องจะแม่นขึ้นไหม?

## A4. กิจกรรมทั้งห้อง: กฎของลีวิตต์จากดาว 54 ดวง
แต่ละคนวัดดาวคนละดวง รวมกันทั้งห้องเราจะได้ **ความสัมพันธ์คาบ–ความสว่าง** ของดาวเซเฟอิดในเมฆแมเจลแลนใหญ่
แบบเดียวกับที่ Henrietta Leavitt พบในปี ค.ศ. 1912 (เธอใช้ดาว 25 ดวงในเมฆแมเจลแลนเล็ก)

1. รันเซลล์ถัดไป แล้ว **พิมพ์ตัวเลข 5 ค่าลง [Google Sheet ของห้อง](https://docs.google.com/spreadsheets/d/1cEKrYaF8-NuVHWXmw7WCLfRbo5VDA6AMTyPRXPMJd78/edit)** — 1 แถวต่อ 1 คน
   (ชื่อดาว · คาบ · ความคลาดเคลื่อนของคาบ · แมกนิจูดเฉลี่ย · ความคลาดเคลื่อนของแมกนิจูด)
2. รอเพื่อนส่งผล แล้วรันเซลล์ "ผลของทั้งห้อง" — โน้ตบุ๊กจะดาวน์โหลดข้อมูลของทั้งห้องจาก Sheet มาให้
   ยิ่งเพื่อนส่งมาก ผลยิ่งแม่น — รันเซลล์นั้นซ้ำได้เรื่อย ๆ เมื่อมีข้อมูลเพิ่ม

In [ ]:
span = t.max() - t.min()
period_err = period ** 2 / span                 # ความละเอียดของ periodogram: Δความถี่ ≈ 1 / ช่วงเวลาที่สังเกต → ΔP ≈ P² / ช่วงเวลา
mag_err = np.std(y) / np.sqrt(len(y))           # ความคลาดเคลื่อนของค่าเฉลี่ย = ส่วนเบี่ยงเบนมาตรฐาน / √จำนวนจุด
print("พิมพ์ลง Google Sheet ของห้อง (1 แถวต่อ 1 คน):")
print(f"STAR_NAME = {STAR}\nPERIOD = {period:.4f}\nPERIOD_ERR = {period_err:.4f}\nIMAG = {mean_mag:.3f}\nIMAG_ERR = {mag_err:.3f}")

In [ ]:
SHEET_CSV = "https://docs.google.com/spreadsheets/d/1cEKrYaF8-NuVHWXmw7WCLfRbo5VDA6AMTyPRXPMJd78/export?format=csv"   # ผลของทั้งห้อง

try:
    raw = pd.read_csv(SHEET_CSV)
    raw.columns = [c.strip().upper() for c in raw.columns]
    cls = pd.DataFrame({"star": raw.STAR_NAME.astype(str).str.strip().str.upper(),
                        "P": pd.to_numeric(raw.PERIOD, errors="coerce"), "m": pd.to_numeric(raw.IMAG, errors="coerce")})
    cls = cls[(cls.P > 1) & (cls.P < 40) & (cls.m > 10) & (cls.m < 18)].drop_duplicates("star", keep="last")
except Exception as e:
    print(f"อ่าน Google Sheet ไม่ได้ ({type(e).__name__})"); cls = pd.DataFrame(columns=["star", "P", "m"])
print(f"ข้อมูลจาก Google Sheet ของห้อง: {len(cls)} ดวง")
assert len(cls) >= 3, "ข้อมูลในชีตยังไม่ถึง 3 ดวง — รอเพื่อนส่งผลแล้วรันเซลล์นี้ใหม่"

# ✏️ กฎของลีวิตต์: ฟิตเส้นตรง m = slope × (log₁₀P − 1) + zp  — np.polyfit(x, y, 1) คืนค่า (ความชัน, จุดตัดแกน)
slope, zp = ____
scatter = np.std(cls.m - (slope * (np.log10(cls.P) - 1) + zp))

plt.figure(figsize=(8, 5.5))
plt.scatter(np.log10(cls.P), cls.m, s=40, color="tab:blue", edgecolor="k", label=f"our class ({len(cls)} Cepheids)")
xx = np.linspace(0.3, 1.45, 50)
plt.plot(xx, slope * (xx - 1) + zp, color="tab:orange", lw=2, label=f"fit: slope = {slope:.2f}")
if MY_NUMBER > 0:
    plt.plot(np.log10(period), mean_mag, "*", ms=22, color="gold", mec="k", label=f"my star ({STAR})")
plt.gca().invert_yaxis()
plt.xlabel("log10 period (days)"); plt.ylabel("mean I magnitude")
plt.legend(); plt.title("Leavitt law in the Large Magellanic Cloud"); plt.show()
print(f"ความชัน = {slope:.2f} (ทางช้างเผือก −2.81) | ดาวคาบ 10 วันมีแมกนิจูดเฉลี่ย {zp:.2f} | ความกระจาย {scatter:.2f} แมกนิจูด")

ดาวคาบ 10 วัน (log₁₀P = 1) มีแมกนิจูดสัมบูรณ์ **M_I = −4.76** (จากสูตรในขั้น A3) — เทียบกับแมกนิจูดที่เราเห็น `zp` ได้ระยะทางของทั้งกาแล็กซี

In [ ]:
d_class = ____        # ✏️ ระยะทาง (พาร์เซก): m = zp, M = −4.76 แล้วใช้สูตรเดียวกับขั้น A3
print(f"ระยะทางไปเมฆแมเจลแลนใหญ่จากทั้งห้อง ≈ {d_class / 1000:.1f} กิโลพาร์เซก (A_I = {A_I}) — ค่ามาตรฐาน 49.6 กิโลพาร์เซก")

🔎 **คำถามชวนคิด**
1. ความชันของห้องเราใกล้ −2.81 ของดาวเซเฟอิดในทางช้างเผือกไหม? ถ้าใกล้ แปลว่าอะไร?
2. ระยะทางจากทั้งห้องใกล้ 49.6 กิโลพาร์เซกกว่าระยะทางจากดาวของคุณดวงเดียวไหม? ทำไม?
3. ทำไมจุดจึงไม่อยู่บนเส้นตรงพอดี? (ใบ้: ฝุ่นแต่ละทิศไม่เท่ากัน · เมฆแมเจลแลนใหญ่มีความหนา · ดาวแต่ละดวงร้อนไม่เท่ากัน)

# ส่วน B: ดาวเคราะห์ผ่านหน้าดาวแม่ (transit) จากกล้อง TESS
ดาว **HD 209458** ในกลุ่มดาวม้าบิน (ห่าง 48 พาร์เซก) มีดาวเคราะห์ **HD 209458 b** — ดาวเคราะห์ดวงแรกที่นักดาราศาสตร์เห็นมัน
**ผ่านหน้าดาวแม่** (ค.ศ. 1999) · ข้อมูลจากกล้องโทรทรรศน์อวกาศ **TESS** ซึ่งวัดความสว่างทุก 2 นาที ต่อเนื่องราว 27 วัน (1 sector)

In [ ]:
tt, f, ferr, kind = load_lightcurve(TRANSIT_FILE)
T_REF = tt.min()
tt = tt - T_REF                                        # เริ่มนับเวลาจาก 0
tt, fd, trend = detrend(tt, f, window=721)             # 721 จุด × 2 นาที ≈ 1 วัน: ยาวกว่าการผ่านหน้า (~3 ชั่วโมง) มาก

fig, ax = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
ax[0].plot(tt, fd * trend, ".", ms=1, color="k"); ax[0].plot(tt, trend, color="tab:orange")
ax[0].set(ylabel="relative flux", title="HD 209458: TESS light curve and slow trend")
ax[1].plot(tt, (fd - 1) * 1e3, ".", ms=1, color="k")
ax[1].set(xlabel="time (days)", ylabel="detrended flux (ppt)")
plt.show()
print(f"{len(tt):,} จุด ตลอด {tt.max():.1f} วัน")

กราฟบน: ความสว่างค่อย ๆ เปลี่ยน (จากตัวดาวและกล้อง) — เส้นสีส้มคือ **แนวโน้มช้า ๆ** ที่เราหารออก
กราฟล่าง: เหลือแต่การหรี่สั้น ๆ — เห็นกี่ครั้ง? ห่างกันกี่วัน? ช่องว่างกลางกราฟคือช่วงที่ TESS ส่งข้อมูลกลับโลก

## B1. หาคาบด้วย Box Least Squares (BLS)
คล้าย periodogram แต่ค้นหารูป "หลุมสี่เหลี่ยม" ที่ซ้ำเป็นคาบ ซึ่งเหมาะกับการผ่านหน้ามากกว่า

In [ ]:
periods = np.exp(np.linspace(np.log(0.35), np.log(15), 40000))            # คาบ 0.35–15 วัน
durations = [0.02, 0.03, 0.05, 0.08, 0.12, 0.16, 0.2, 0.25, 0.3]          # ช่วงเวลาที่ผ่านหน้า (วัน)
bls = BoxLeastSquares(tt, fd)
res = bls.power(periods, durations, oversample=5)
i = np.argmax(res.power)
P, t0, depth, dur = res.period[i], res.transit_time[i], res.depth[i], res.duration[i]

plt.figure(figsize=(12, 3.5))
plt.plot(res.period, res.power, lw=0.8); plt.axvline(P, color="tab:orange", ls="--")
plt.xscale("log"); plt.xlabel("period (days)"); plt.ylabel("BLS power"); plt.show()
print(f"คาบ = {P:.4f} วัน | ความลึก = {depth * 100:.2f}% | นาน ≈ {dur * 24:.1f} ชั่วโมง")

🔎 ยอดเล็ก ๆ ที่คาบ P/2 และ 2P คืออะไร? (ใบ้: พับกราฟด้วยคาบครึ่งหนึ่งหรือสองเท่า การหรี่ก็ยังซ้อนกันได้บางส่วน)

## B2. พับกราฟแสงรอบการผ่านหน้า

In [ ]:
hours = ((tt - t0 + P / 2) % P - P / 2) * 24             # เวลานับจากกึ่งกลางการผ่านหน้า (ชั่วโมง)
bins = np.arange(-6, 6.01, 0.25)
idx = np.digitize(hours, bins)
bx = [hours[idx == k].mean() for k in range(1, len(bins)) if np.any(idx == k)]
by = [(fd[idx == k].mean() - 1) * 100 for k in range(1, len(bins)) if np.any(idx == k)]

plt.figure(figsize=(9, 4.5))
plt.plot(hours, (fd - 1) * 100, ".", ms=2, color="lightgray", label="every 2 minutes")
plt.plot(bx, by, "o-", color="tab:orange", label="average every 15 minutes")
plt.axhline(-depth * 100, color="k", ls=":", label=f"BLS depth {depth * 100:.2f}%")
plt.xlim(-6, 6); plt.xlabel("hours from mid-transit"); plt.ylabel("brightness change (%)")
plt.legend(); plt.title(f"HD 209458 b folded at P = {P:.4f} d"); plt.show()

🔎 ก้นของการผ่านหน้าโค้ง ไม่แบนเป็นกล่อง — เพราะขอบของดาวมืดกว่าตรงกลาง (**limb darkening**)
ความลึกแบบ "กล่อง" ของ BLS จึงน้อยกว่าความลึกจริงเล็กน้อย

## B3. ดาวเคราะห์ใหญ่แค่ไหน และโคจรห่างแค่ไหน?
ความลึกของการผ่านหน้า = สัดส่วนพื้นที่ที่ถูกบัง: **depth = (R_ดาวเคราะห์ / R_ดาว)²**
และกฎข้อที่ 3 ของเคปเลอร์: **a³ = M × P²** (a เป็น AU, M เป็นมวลดวงอาทิตย์, P เป็นปี)

In [ ]:
R_STAR = 1.19                          # 🔧 รัศมีดาวแม่ (เท่าของดวงอาทิตย์) — NASA Exoplanet Archive
M_STAR = 1.23                          # 🔧 มวลดาวแม่ (เท่าของดวงอาทิตย์)

rp = ____                     # ✏️ รัศมีดาวเคราะห์ (เท่าของดวงอาทิตย์)
a_au = ____    # ✏️ กฎข้อ 3 ของเคปเลอร์ (P เป็นปี)
rp_earth = rp * 109.1                              # 1 รัศมีดวงอาทิตย์ = 109.1 รัศมีโลก = 9.73 รัศมีดาวพฤหัสบดี
print(f"R_p/R_* = {np.sqrt(depth):.3f} | รัศมี ≈ {rp_earth:.1f} เท่าของโลก ({rp * 9.73:.2f} เท่าของดาวพฤหัสบดี)")
print(f"วงโคจร ≈ {a_au:.3f} AU (ดาวพุธ 0.39 AU) — ค่าที่ตีพิมพ์: รัศมี 15.58 เท่าของโลก")

🔎 เทียบ: เนปจูน 3.9 เท่าของโลก · ดาวเสาร์ 9.4 เท่า · ดาวพฤหัสบดี 11.2 เท่า — ดาวเคราะห์ดวงนี้คล้ายดาวดวงไหนในระบบสุริยะ?
อยู่ใกล้ดาวแม่กว่าดาวพุธกี่เท่า? คิดว่าร้อนแค่ไหน?

## B4. ดาวเคราะห์ดวงนี้ลอยน้ำได้ไหม?
มวลของดาวเคราะห์วัดจากการที่มันดึงดาวแม่ให้ส่ายไปมา (ความเร็วแนวเล็ง — หลักการเดียวกับโน้ตบุ๊ก 02 ขั้น A5)
นักดาราศาสตร์วัดมวลของ HD 209458 b ได้ **232 เท่าของโลก** (0.73 เท่าของดาวพฤหัสบดี)

In [ ]:
MP = 232.0                              # มวลดาวเคราะห์ (เท่าของโลก) — NASA Exoplanet Archive
rho = ____                # ✏️ ความหนาแน่น (g/cm³): โลก 5.51 g/cm³, MP และ rp_earth เป็นหน่วยของโลก
print(f"ความหนาแน่น ≈ {rho:.2f} g/cm³ (น้ำ 1.0 · ดาวเสาร์ 0.69 · โลก 5.51)")

🔎 น้ำมีความหนาแน่น 1 g/cm³ — ดาวเคราะห์ดวงนี้ "ลอยน้ำได้" ไหม? แปลว่ามันประกอบด้วยอะไรเป็นส่วนใหญ่?
ในภารกิจช่วงบ่าย แต่ละกลุ่มจะทำขั้นตอนเดียวกันนี้กับดาวเคราะห์ของตัวเอง

## B5. (ท้าทาย) ถ้าดาวเคราะห์เล็กลง ยังหาเจอไหม?
ทดลองกับ **ข้อมูลจำลอง** แบบกล้อง TESS: ดาวเคราะห์คาบ 3.5247 วัน

In [ ]:
# 🔧 ลองเปลี่ยน rp_rs: 0.12 (ดาวพฤหัสบดี), 0.035 (ดาวเนปจูน), 0.01 (โลก)
t2, f2, e2 = synthetic_transit(rp_rs=0.035)
p2 = np.exp(np.linspace(np.log(0.5), np.log(10), 20000))
r2 = BoxLeastSquares(t2, f2, e2).power(p2, [0.05, 0.08, 0.1, 0.125, 0.15])
j = np.argmax(r2.power)
print(f"คาบที่หาได้ = {r2.period[j]:.4f} วัน (คาบจริง 3.5247) | ความลึก = {r2.depth[j] * 1e6:.0f} ppm"
      f" | S/N ของความลึก = {r2.depth_snr[j]:.1f} (ต่ำกว่า ~7 = เชื่อถือไม่ได้)")

🔎 **คำถามชวนคิด**
1. ทำไมดาวเคราะห์ขนาดเท่าโลกจึงหายากกว่าดาวพฤหัสบดีมาก?
2. ถ้าดาวเคราะห์มีคาบ 1 ปี (เหมือนโลก) ข้อมูล 27 วันของ TESS จะพอไหม?
3. ทำไมการวัดทั้ง "ขนาด" (การผ่านหน้า) และ "มวล" (ความเร็วแนวเล็ง) จึงสำคัญมาก?